In [ ]:
#Setup e Caricamento del File
!pip install pandas scipy

from google.colab import files
import json
import pandas as pd
import scipy.stats as st
import numpy as np
import io

uploaded = files.upload()

file_name = list(uploaded.keys())[0]
print(f"File '{file_name}' caricato con successo.")

Carica il tuo file 'results.json' (potrebbe richiedere tempo)...


Saving results.json to results.json
File 'results.json' caricato con successo.


In [ ]:
#Estrazione, Trasformazione e Caricamento

#Caricamento del file JSON in memoria
data = json.load(io.BytesIO(uploaded[file_name]))

all_scalar_data = []

#Dizionario per tradurre i nomi  (basato sull'omnetpp.ini)
#$0=m(SS1), $1=m(SS2) [da Config1/2]
#$2=m (interArrivalTime), $3=q (userTypeProbability), $4=p (feedbackProbability)
PARAM_MAP = {
    '$0': 'm_SS1',
    '$1': 'm_SS2',
    '$2': 'm_arrivi',
    '$3': 'q_prob_U1',
    '$4': 'p_feedback'
}

for run_id, run_data in data.items():
    attributes = run_data.get('attributes', {})
    config_name = attributes.get('experiment', 'N/A')

    try:
        repetition = int(attributes.get('repetition', -1))
    except ValueError:
        repetition = -1

    itervars_str = attributes.get('iterationvars', '')
    params = {}
    if itervars_str:
        try:
            pairs = itervars_str.split(', ')
            for pair in pairs:
                key, value = pair.split('=')
                #Traduzione del nome della chiave se esiste nella mappa
                col_name = PARAM_MAP.get(key, key)
                params[col_name] = value.replace('s', '')
        except Exception:
            pass

    scalars = run_data.get('scalars', [])
    for scalar in scalars:
        metric_name = scalar.get('name')

        metric_value = scalar.get('value')

        #Un 'null' per timeavg significa che la coda
        #è rimasta vuota, sostituzione con valore medio 0.0
        if metric_value is None:
            metric_value = 0.0

        #Aggiunta delle metriche necessarie
        if metric_name and metric_name.startswith('queueLen'):
            row = {
                'run_id': run_id,
                'Config': config_name,
                'repetition': repetition,
                'metric_name': metric_name,
                'value': float(metric_value)
            }
            row.update(params)
            all_scalar_data.append(row)

df = pd.DataFrame(all_scalar_data)
display(df.head())

Caricamento del file JSON in memoria...
Trasformazione dei dati (ETL) per 9720 run...

Trasformazione completata.
Creato un DataFrame con 38880 righe di dati validi.

Anteprima dei dati (con colonne rinominate e null=0):


,run_id,Config,repetition,metric_name,value,m_SS1,m_SS2,m_arrivi,q_prob_U1,p_feedback
0,Config2-1024-20251102-19:53:15-7464,Config2,4,queueLenNode1_U2:timeavg,1709.273193,2.4,3.0,6.0,0.8,0.6
1,Config2-1024-20251102-19:53:15-7464,Config2,4,queueLenNode1_U1:timeavg,30.173795,2.4,3.0,6.0,0.8,0.6
2,Config2-1024-20251102-19:53:15-7464,Config2,4,queueLenSS2:timeavg,0.008527,2.4,3.0,6.0,0.8,0.6
3,Config2-1024-20251102-19:53:15-7464,Config2,4,queueLenSS1:timeavg,0.196336,2.4,3.0,6.0,0.8,0.6
4,Config2-993-20251102-19:53:13-7431,Config2,13,queueLenNode1_U2:timeavg,3792.621404,2.4,3.0,6.0,0.6,0.8


In [ ]:
#Calcolo Metrica 1: Lunghezza Code

#Definizione delle colonne di configurazione
parametri_configurazione = ['Config', 'm_SS1', 'm_SS2', 'm_arrivi', 'q_prob_U1', 'p_feedback']

#Riorganizzazione dati (Pivot)
df_pivot = df.pivot_table(
    index=parametri_configurazione + ['repetition'],
    columns='metric_name',
    values='value'
).reset_index()

#Calcolo metriche totali (U1+U2)
df_pivot['CodaTotale_Nodo1'] = df_pivot['queueLenNode1_U1:timeavg'] + df_pivot['queueLenNode1_U2:timeavg']
df_pivot['CodaTotale_Nodo2'] = df_pivot['queueLenSS1:timeavg'] + df_pivot['queueLenSS2:timeavg']

#Calcolo Stima Puntuale (Media) e Intervallo di Confidenza
grouped = df_pivot.groupby(parametri_configurazione)

#Valore critico 't' per il 95% di confidenza con (n=20 -> df=19)
T_VALUE = st.t.ppf(0.975, 19)

#Calcolo statistiche aggregate
stima_puntuale = grouped.mean()
deviazione_standard = grouped.std()
n_campioni = grouped.count()

#Calcolo del margine di errore per l'Intervallo di Confidenza
#Formula: t * (s / sqrt(n))
margine_errore_ci = T_VALUE * (deviazione_standard / np.sqrt(n_campioni))

ci_limite_inferiore = stima_puntuale - margine_errore_ci
ci_limite_superiore = stima_puntuale + margine_errore_ci

#Rimozione della colonna 'repetition', non più necessaria
stima_puntuale = stima_puntuale.drop(columns='repetition')
ci_limite_inferiore = ci_limite_inferiore.drop(columns='repetition')
ci_limite_superiore = ci_limite_superiore.drop(columns='repetition')

print("\nStima Puntuale (Media delle 20 repliche):")
display(stima_puntuale)

print("\nIntervallo di Confidenza (95%) - Limite Inferiore:")
display(ci_limite_inferiore)

print("\nIntervallo di Confidenza (95%) - Limite Superiore:")
display(ci_limite_superiore)

Inizio calcolo statistiche...
Riorganizzazione dati (Pivot)...
Calcolo metriche totali (U1+U2)...
Calcolo Stima Puntuale (Media) e Intervallo di Confidenza...

--- Analisi Metrica 1 Completata ---

Stima Puntuale (Media delle 20 repliche):


metric_name                                        queueLenNode1_U1:timeavg  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                             
Config1 2.4   2.0   4.0      0.4       0.6                         0.665835   
                                       0.8                        75.843016   
                                       0.9                       978.840079   
                             0.6       0.6                         2.147434   
                                       0.8                      2745.313850   
...                                                                     ...   
Config2 3.5   4.0   6.0      0.6       0.8                      1857.108483   
                                       0.9                      2898.195122   
                             0.8       0.6                       100.705306   
                                       0.8                      3689.615316   
                                       0.9                      4704.623195   

metric_name                                        queueLenNode1_U2:timeavg  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                             
Config1 2.4   2.0   4.0      0.4       0.6                      3875.961275   
                                       0.8                      8200.788356   
                                       0.9                      8244.342495   
                             0.6       0.6                      3318.144631   
                                       0.8                      5516.891513   
...                                                                     ...   
Config2 3.5   4.0   6.0      0.6       0.8                      3681.656654   
                                       0.9                      3679.001968   
                             0.8       0.6                      1775.530902   
                                       0.8                      1840.694740   
                                       0.9                      1831.555481   

metric_name                                        queueLenSS1:timeavg  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                        
Config1 2.4   2.0   4.0      0.4       0.6                    0.167333   
                                       0.8                   12.654626   
                                       0.9                 1785.159099   
                             0.6       0.6                    0.490062   
                                       0.8                   14.939035   
...                                                                ...   
Config2 3.5   4.0   6.0      0.6       0.8                    7.289693   
                                       0.9                  807.505134   
                             0.8       0.6                    0.988205   
                                       0.8                    7.721041   
                                       0.9                  842.712676   

metric_name                                        queueLenSS2:timeavg  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                        
Config1 2.4   2.0   4.0      0.4       0.6                    0.056683   
                                       0.8                    0.001347   
                                       0.9                    0.000000   
                             0.6       0.6                    0.019961   
                                       0.8                    0.000000   
...                                                                ...   
Config2 3.5   4.0   6.0      0.6       0.8                    0.000000   
                                       0.9                    0.000000   
                             0.8       0.6                    0.001315   
                                       0.8                    0.000000   
                                       0.9                    0.000000   

metric_name       


Intervallo di Confidenza (95%) - Limite Inferiore:


metric_name                                        queueLenNode1_U1:timeavg  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                             
Config1 2.4   2.0   4.0      0.4       0.6                         0.648011   
                                       0.8                        47.459903   
                                       0.9                       899.160092   
                             0.6       0.6                         2.087553   
                                       0.8                      2693.561241   
...                                                                     ...   
Config2 3.5   4.0   6.0      0.6       0.8                      1819.729396   
                                       0.9                      2841.143711   
                             0.8       0.6                        71.825596   
                                       0.8                      3641.637754   
                                       0.9                      4644.277509   

metric_name                                        queueLenNode1_U2:timeavg  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                             
Config1 2.4   2.0   4.0      0.4       0.6                      3818.048701   
                                       0.8                      8144.124423   
                                       0.9                      8202.374684   
                             0.6       0.6                      3258.007791   
                                       0.8                      5473.126760   
...                                                                     ...   
Config2 3.5   4.0   6.0      0.6       0.8                      3657.171590   
                                       0.9                      3649.653540   
                             0.8       0.6                      1745.513677   
                                       0.8                      1819.398814   
                                       0.9                      1813.000839   

metric_name                                        queueLenSS1:timeavg  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                        
Config1 2.4   2.0   4.0      0.4       0.6                    0.160344   
                                       0.8                   10.766398   
                                       0.9                 1694.296807   
                             0.6       0.6                    0.474863   
                                       0.8                   12.555717   
...                                                                ...   
Config2 3.5   4.0   6.0      0.6       0.8                    6.611204   
                                       0.9                  748.383209   
                             0.8       0.6                    0.953230   
                                       0.8                    6.892587   
                                       0.9                  779.844472   

metric_name                                        queueLenSS2:timeavg  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                        
Config1 2.4   2.0   4.0      0.4       0.6                    0.054351   
                                       0.8                    0.000084   
                                       0.9                    0.000000   
                             0.6       0.6                    0.018606   
                                       0.8                    0.000000   
...                                                                ...   
Config2 3.5   4.0   6.0      0.6       0.8                    0.000000   
                                       0.9                    0.000000   
                             0.8       0.6                   -0.000075   
                                       0.8                    0.000000   
                                       0.9                    0.000000   

metric_name       


Intervallo di Confidenza (95%) - Limite Superiore:


metric_name                                        queueLenNode1_U1:timeavg  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                             
Config1 2.4   2.0   4.0      0.4       0.6                         0.683658   
                                       0.8                       104.226130   
                                       0.9                      1058.520066   
                             0.6       0.6                         2.207315   
                                       0.8                      2797.066459   
...                                                                     ...   
Config2 3.5   4.0   6.0      0.6       0.8                      1894.487570   
                                       0.9                      2955.246533   
                             0.8       0.6                       129.585016   
                                       0.8                      3737.592878   
                                       0.9                      4764.968881   

metric_name                                        queueLenNode1_U2:timeavg  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                             
Config1 2.4   2.0   4.0      0.4       0.6                      3933.873850   
                                       0.8                      8257.452289   
                                       0.9                      8286.310305   
                             0.6       0.6                      3378.281470   
                                       0.8                      5560.656265   
...                                                                     ...   
Config2 3.5   4.0   6.0      0.6       0.8                      3706.141718   
                                       0.9                      3708.350396   
                             0.8       0.6                      1805.548127   
                                       0.8                      1861.990665   
                                       0.9                      1850.110124   

metric_name                                        queueLenSS1:timeavg  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                        
Config1 2.4   2.0   4.0      0.4       0.6                    0.174321   
                                       0.8                   14.542855   
                                       0.9                 1876.021392   
                             0.6       0.6                    0.505261   
                                       0.8                   17.322354   
...                                                                ...   
Config2 3.5   4.0   6.0      0.6       0.8                    7.968182   
                                       0.9                  866.627060   
                             0.8       0.6                    1.023179   
                                       0.8                    8.549495   
                                       0.9                  905.580880   

metric_name                                        queueLenSS2:timeavg  \
Config  m_SS1 m_SS2 m_arrivi q_prob_U1 p_feedback                        
Config1 2.4   2.0   4.0      0.4       0.6                    0.059016   
                                       0.8                    0.002611   
                                       0.9                    0.000000   
                             0.6       0.6                    0.021317   
                                       0.8                    0.000000   
...                                                                ...   
Config2 3.5   4.0   6.0      0.6       0.8                    0.000000   
                                       0.9                    0.000000   
                             0.8       0.6                    0.002704   
                                       0.8                    0.000000   
                                       0.9                    0.000000   

metric_name       

In [ ]:
# Download dei Risultati

#Salvataggio dei file CSV
stima_puntuale.to_csv("metrica1_stima_puntuale.csv")
ci_limite_inferiore.to_csv("metrica1_ci_inferiore.csv")
ci_limite_superiore.to_csv("metrica1_ci_superiore.csv")

files.download("metrica1_stima_puntuale.csv")
files.download("metrica1_ci_inferiore.csv")
files.download("metrica1_ci_superiore.csv")



Preparazione file per il download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download avviato.
